# KWISMO — Collecte de donnees (scraping + OCR)

Ce notebook n'ecrit aucune logique lui-meme : il appelle uniquement le code de `src/data/` (scrape.py, scrape_social.py, ocr.py, metrics.py). Le projet exige Python 3.13 partout (voir `src/__init__.py`) ; comme Colab fournit Python 3.12 par defaut et qu'on ne peut pas demander a chacun de changer son runtime, les cellules ci-dessous installent automatiquement un Python 3.13 isole (venv) et executent tout le code `src/` a travers lui — sans toucher au noyau Colab. En local, ton venv est deja en 3.13, donc rien de special ne se passe.

In [ ]:
# Detecte si on execute sur Google Colab ou en local.
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

print("Environnement :", "Colab" if ON_COLAB else "local")

In [ ]:
# Sur Colab : clone le depot (une seule fois, chemin fixe) et s'y place.
# En local : on est deja dans le depot, rien a cloner.
import os
from pathlib import Path

if ON_COLAB:
    PROJECT_DIR = Path("/content/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /content/kwismo
    else:
        !git -C /content/kwismo fetch && git -C /content/kwismo reset --hard origin/main
    os.chdir(PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd()

print("Dossier de travail :", PROJECT_DIR)

## Python 3.13 isole (Colab uniquement)

Cree un venv avec Python 3.13 (deadsnakes) a cote du venv Colab par defaut, sans y toucher, puis y installe `requirements.txt`. Tout le code `src/` tourne ensuite via cet interpreteur (sous-processus), jamais dans le noyau du notebook — c'est ce qui evite de demander a l'utilisateur de changer son runtime Colab.

In [ ]:
# Sur Colab : installe Python 3.13 (absent par defaut) et toutes les dependances (requirements.txt + Playwright) dans un venv isole.
# En local : Python 3.13 est deja l'interpreteur courant, rien a installer.
import sys

VENV_DIR = PROJECT_DIR / ".venv313"

if ON_COLAB:
    if not (VENV_DIR / "bin" / "python").exists():
        !apt-get -qq install -y software-properties-common > /dev/null
        !add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
        !apt-get -qq update > /dev/null
        !apt-get -qq install -y python3.13 python3.13-venv > /dev/null
        !python3.13 -m venv {VENV_DIR}
        !{VENV_DIR}/bin/pip install -q --upgrade pip
        !{VENV_DIR}/bin/pip install -q -r requirements.txt
        !{VENV_DIR}/bin/python -m playwright install --with-deps chromium
    PYTHON_BIN = str(VENV_DIR / "bin" / "python")
else:
    PYTHON_BIN = sys.executable

print("Interpreteur utilise pour src/ :", PYTHON_BIN)

## Configuration

`.env` n'est jamais versionne. En local il existe deja. Sur Colab, recree-le (comptes de scraping optionnels — sans eux, `scrape_social.run()` ne fait simplement rien).

In [ ]:
# Sur Colab : recree un .env minimal pour la session (jamais versionne).
# En local : le .env existe deja, on ne touche a rien.
if ON_COLAB:
    env_lines = [
        'MODEL_DIR="./models"',
        'HF_MODEL_NAME="Davlan/afro-xlmr-base"',
        '# FACEBOOK_ACCOUNTS="user1:pass1,user2:pass2"  # decommenter si besoin (voir Colab Secrets)',
    ]
    Path(".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
    print(".env cree pour la session Colab.")
else:
    print("Local : .env existe deja, rien a faire.")

In [ ]:
# Petite fonction utilitaire : lance `python -m <module>` avec le Python 3.13
# (PYTHON_BIN), en sous-processus, pour chaque etape ci-dessous.
import subprocess


def run_module(module: str) -> None:
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a echoue (code {result.returncode})")

## 1. Sources texte (decouverte dynamique via recherche web)

In [ ]:
# Cherche des pages via mots-cles, extrait texte + images, dedoublonne.
run_module("src.data.scrape")

## 2. Facebook / Instagram / X (si des comptes sont configures dans .env)

In [ ]:
# Se connecte aux comptes configures (FACEBOOK_ACCOUNTS, ...) et collecte.
run_module("src.data.scrape_social")

## 3. OCR des captures collectees

In [ ]:
# Extrait le texte des captures d'ecran collectees (EasyOCR).
run_module("src.data.ocr")

## 4. Metriques — evolution de la collecte

In [ ]:
# Met a jour l'historique des executions et regenere les graphes.
run_module("src.data.metrics")

In [ ]:
# Affiche les graphes generes par l'etape precedente.
from IPython.display import Image, display

plots_dir = PROJECT_DIR / "data" / "interim" / "metrics" / "plots"
display(Image(filename=str(plots_dir / "evolution_collecte.png")))
display(Image(filename=str(plots_dir / "erreurs_par_run.png")))

## 5. Publier les donnees collectees (Git)

`data/raw/scraped/messages.jsonl` et `data/interim/metrics/` sont versionnes — un `git push` les rend disponibles a toute l'equipe (voir `COLAB.md`).

In [ ]:
SAVE_METHOD = "drive"

if ON_COLAB:
    if SAVE_METHOD == "drive":
        from google.colab import drive
        from pathlib import Path
        import shutil
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/kwismo_data")
        dest.mkdir(parents=True, exist_ok=True)
        if Path("data/raw/scraped/messages.jsonl").exists():
            shutil.copy("data/raw/scraped/messages.jsonl", dest / "messages.jsonl")
        if Path("data/interim/known_domains.json").exists():
            shutil.copy("data/interim/known_domains.json", dest / "known_domains.json")
        if Path("data/raw/scraped/images").exists():
            shutil.copytree("data/raw/scraped/images", dest / "images", dirs_exist_ok=True)
        if Path("data/interim/metrics").exists():
            shutil.copytree("data/interim/metrics", dest / "metrics", dirs_exist_ok=True)
        print("Sauvegarde Google Drive terminee :", dest)
    elif SAVE_METHOD == "git":
        !git config user.email "toi@exemple.com"
        !git config user.name "Ton Nom"
        !git add data/raw/scraped/*.jsonl data/interim/metrics/ data/interim/known_domains.json
        !git commit -m "Collecte de donnees - session Colab"
